In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

In [ ]:
file_path = "/kaggle/input/datasets/sv1802/zillow-rental-market-data-california-2025/Zillow Rent Data.xlsx - Data.csv"
df = pd.read_csv(file_path, low_memory=False)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
#All column names
#df.columns.tolist()[:300] 

In [ ]:
#Grouping column names with the first word
df.columns.str.split("/").str[0].value_counts().head(5)

# 1. Drop photos, links, and IDs columns
163 of the 300 columns are photo links (`carouselPhotos`) which are useless for analysis.

In [ ]:
#Every column with "carouselPhotos" in its name
photo_cols = df.filter(like="carouselPhotos").columns   
len(photo_cols)

In [ ]:
#`carouselPhotos` are just image links
df[["carouselPhotos/0/url", "carouselPhotos/1/url"]].head(2)  

Links and IDs point to a page or name a listing, but they say nothing about rent.

In [ ]:
df[["detailUrl", "imgSrc", "id", "providerListingId", "zpid"]].head(3)

In [ ]:
link_cols = ["detailUrl", "imgSrc", "foundFromSearchUrl", "streetViewURL", "streetViewMetadataURL",
             "providerListingId", "id", "zpid"]

df = df.drop(columns=list(photo_cols) + link_cols)
df.shape

# 2. Drop low-information & redundant columns
1. **Constant columns:** Columns where every row has 0 or 1 unique value (no variance across listings).
2. **Duplicate nested columns:** `hdpData/homeInfo/...` fields that repeat existing top-level attributes (like city, zip, beds, baths, area).
3. **Frontend metadata:** UI badges, search page flags, and bathroom detail breakdowns not required for modeling.

In [ ]:
#Columns with 0 or 1 unique values including checks for NaNs. 
#Because by default, pandas ignores missing (NaN) values when counting unique entries.
#one_value = df.columns[df.nunique() <= 1]

one_value = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
len(one_value)
df = df.drop(columns=one_value)

In [ ]:
df.shape

The nested `hdpData/homeInfo/...` columns look like copies of the top-level ones (city, zip, latitude, price, beds, baths, area). 

In [ ]:
#Checking if they are actually copies
pairs = [("addressCity", "hdpData/homeInfo/city"),
         ("addressZipcode", "hdpData/homeInfo/zipcode"),
         ("latLong/latitude", "hdpData/homeInfo/latitude"),
         ("unformattedPrice", "hdpData/homeInfo/price"),
         ("beds", "hdpData/homeInfo/bedrooms"),
         ("baths", "hdpData/homeInfo/bathrooms"),
         ("area", "hdpData/homeInfo/livingArea"),
         ("zestimate", "hdpData/homeInfo/zestimate")]

for top, nested in pairs:
    #Rows where both columns have a value
    both = df[top].notna() & df[nested].notna() 
    #Share of those rows that match
    same = (df.loc[both, top] == df.loc[both, nested]).mean() 
    print(top, "vs", 
          nested, " - same in", 
          same * 100, "% of rows | blanks:", 
          df[top].isna().sum(), "vs", 
          df[nested].isna().sum())

They match in 100% of rows, so they carry no new information. 
The nested copy often has more blanks (for example latitude: 692 vs 88), so we keep the top-level column.

`price` is also a repeat, just written as text (`"$2,295+/mo"` instead of `2295`):

In [ ]:
df[["price", "unformattedPrice"]].dropna().head(3)

In [ ]:
repeat_cols = ["hdpData/homeInfo/city", "hdpData/homeInfo/zipcode", "hdpData/homeInfo/latitude",
               "hdpData/homeInfo/longitude", "hdpData/homeInfo/streetAddress", "hdpData/homeInfo/price",
               "hdpData/homeInfo/bedrooms", "hdpData/homeInfo/bathrooms", "hdpData/homeInfo/livingArea",
               "hdpData/homeInfo/state", "hdpData/homeInfo/zpid", "price"]

df = df.drop(columns=[c for c in repeat_cols if c in df.columns])
df.shape

In [ ]:
#Columns not required for analysis like `factAndFeatures`, `marketingTreatments` must be dropped
meta_data_cols = ['factsAndFeatures/fullBathroomCount',
 'factsAndFeatures/halfBathroomCount',
 'factsAndFeatures/hasAirConditioning',
 'factsAndFeatures/hasFireplace',
 'factsAndFeatures/hasPool',
 'factsAndFeatures/hasSpa',
 'factsAndFeatures/quarterBathroomCount',
 'factsAndFeatures/threeQuarterBathroomCount','marketingStatusSimplifiedCd',
 'marketingTreatments/0',
 'marketingTreatments/1',
 'marketingTreatments/2',
 'marketingTreatments/3',
 'marketingTreatments/4',
 'marketingTreatments/5',
 'marketingTreatments/6', 'foundOnSearchPage',
 'has3DModel',
 'hasAdditionalAttributions',
 'hasImage',
 'hasOpenHouse',
 'hasVideo']
df = df.drop(columns=[c for c in meta_data_cols if c in df.columns])
df.shape

In [ ]:
#Drop the rest of the columns that show flags, listing level UI & boolean switches, open house fields, multi-unit cols beyond primary unit listing.
#Explicitly keeping only the columns needed for analysis
keep_cols = [
    #Pricing & Availability
    "unformattedPrice",
    "availabilityDate",
    #Location
    "address",
    "addressCity",
    "addressZipcode",
    "latLong/latitude",
    "latLong/longitude",
    #Property attributes
    "beds",
    "baths",
    "area",
    "hdpData/homeInfo/homeType",
    #Zestimate & Market metrics
    "hdpData/homeInfo/rentZestimate",
    "hdpData/homeInfo/zestimate",
    "hdpData/homeInfo/daysOnZillow",
    "hdpData/homeInfo/priceChange",
    #Multi-unit details
    "isBuilding",
    "buildingName",
    "units/0/beds",
    "units/0/price",
    "units/1/beds",
    "units/1/price",
]

df = df[[c for c in keep_cols if c in df.columns]].copy()
df.shape

Rename columns for clarity

In [ ]:
df = df.rename(columns={
    "addressCity": "city",
    "addressZipcode": "zip",
    "hdpData/homeInfo/homeType": "type",
    "unformattedPrice": "rent",
    "hdpData/homeInfo/rentZestimate": "rent_zestimate",
    "hdpData/homeInfo/zestimate": "home_zestimate",
    "hdpData/homeInfo/daysOnZillow": "days_on_zillow",
    "hdpData/homeInfo/priceChange": "price_change",
    "availabilityDate": "availability_date",
    "latLong/latitude": "lat",
    "latLong/longitude": "lon",
    "isBuilding": "is_building",
    "buildingName": "building_name",
    "units/0/beds": "tier1_beds",
    "units/0/price": "tier1_price",
    "units/1/beds": "tier2_beds",
    "units/1/price": "tier2_price",
})
df.head()

# 3. Handling missing values

In [ ]:
df.isna().sum()

`rent`, `beds`, `baths` and `type` are all missing in the same **604 rows**. That is not random. Let's look at those rows.

In [ ]:
df[df["rent"].isna()][["address", "building_name", "tier1_beds", "tier1_price"]].head()

In [ ]:
#Checking if they are all buildings?
df[df["rent"].isna()]["is_building"].value_counts()

Yes. Every row with no rent is an **apartment building**. A building does not have one rent, it lists a *starting price* for each type of unit (`tier1_price` = "$2,499+").

So we have two kinds of rows. We split them into two tables:
- `listings`: one home or apartment with one rent (1,896 rows)
- `buildings`: apartment buildings with starting prices (604 rows)

In [ ]:
listings = df[df["rent"].notna()].copy()
buildings = df[df["rent"].isna()].copy()

print("listings:", listings.shape)
print("buildings:", buildings.shape)

# 4. Cleaning the listings

In [ ]:
listings.info()

In [ ]:
#`availability_date` is text, not a date
listings["availability_date"] = pd.to_datetime(listings["availability_date"])
listings["availability_date"].min(), listings["availability_date"].max()

The dates run from April to November 2025, so this data is a snapshot from 2025.

## Check the city names

In [ ]:
sorted(listings["city"].unique())

In [ ]:
#`Dana Point` and `Dana Pt` are the same city spelled two ways. If we leave it, pandas treats them as two cities.
listings["city"].str.contains("Dana").sum()
listings[listings["city"].str.contains("Dana")]["city"].value_counts()

In [ ]:
listings["city"] = listings["city"].replace("Dana Pt", "Dana Point")
listings[listings["city"].str.contains("Dana")]["city"].value_counts()

## Duplicates

In [ ]:
listings.duplicated().sum()

In [ ]:
#Same address, beds, baths, area and rent = same listing scraped twice
listings.duplicated(subset=["address", "beds", "baths", "area", "rent", "availability_date"]).sum()

## Missing values after cleaning

In [ ]:
listings.isna().sum()

# 5. Rent

In [ ]:
listings["rent"].describe()

In [ ]:
print("mean rent:  ", round(listings["rent"].mean()))
print("median rent:", round(listings["rent"].median()))

The mean is much higher than the median ($4,917 vs $3,600). A few very expensive homes pull the average up. When that happens, the median is the better "typical" number, so we will use the median.

## The most expensive and cheapest rent

In [ ]:
listings[listings["rent"] == listings["rent"].max()][["address", "city", "type", "beds", "baths", "area", "rent"]]

In [ ]:
listings[listings["rent"] == listings["rent"].min()][["address", "city", "type", "beds", "area", "rent"]]

## Outliers (IQR rule)

In [ ]:
Q1 = listings["rent"].quantile(0.25)
Q3 = listings["rent"].quantile(0.75)
IQR = Q3 - Q1
#Anything above this is an outlier
UB = Q3 + 1.5 * IQR   
UB

In [ ]:
outliers = listings[listings["rent"] > UB]
print("outliers:", len(outliers))
outliers["city"].value_counts().head(5)

These are mostly big homes in Newport Beach, Huntington Beach, Laguna Beach and Corona Del Mar. They are real rentals, not errors. So we keep them and use medians.

In [ ]:
#Histogram of rent (only rents under $15,000 so the chart is readable)
listings[listings["rent"] < 15000]["rent"].plot(kind="hist", bins=50, title="Rent distribution")
plt.xlabel("Monthly rent ($)")
plt.show()

# 6. Create a new col: Rent per sq foot

In [ ]:
#A bigger home costs more just because it is bigger. Rent per sq ft lets us compare fairly.
listings["rent_per_sqft"] = listings["rent"] / listings["area"]
listings[["address", "rent", "area", "rent_per_sqft"]].head()

## Clean the buildings : The 604 rows

In [ ]:
#Each building lists up to 4 unit types(for example studio, 1 bed, 2 bed, 3 bed), each with a starting price.
buildings[["building_name", "city", "tier1_beds", "tier1_price", "tier2_beds", "tier2_price"]].head()

In [ ]:
#We want one row per unit type, so we stack the 4 groups of columns on top of each other.
tiers = []
for i in range(1, 3):   # tier1 and tier2
    t = buildings[["city", f"tier{i}_beds", f"tier{i}_price"]].dropna()   # skip buildings without this unit type
    t.columns = ["city", "beds", "starting_price"]
    tiers.append(t)

units = pd.concat(tiers)
units.shape

In [ ]:
#The prices are still text like `$2,499+`. Remove the symbols and convert to numbers.
#Clean the price columns in the `buildings` DataFrame FIRST before concatenating
buildings["tier1_price"] = buildings["tier1_price"].astype(str).str.replace(r"[$,+]", "", regex=True)
buildings["tier2_price"] = buildings["tier2_price"].astype(str).str.replace(r"[$,+]", "", regex=True)

#Stack tier 1 and tier 2 into units
tier1 = buildings[["city", "tier1_beds", "tier1_price"]].dropna()
tier1.columns = ["city", "beds", "starting_price"]

tier2 = buildings[["city", "tier2_beds", "tier2_price"]].dropna()
tier2.columns = ["city", "beds", "starting_price"]

units = pd.concat([tier1, tier2])

#Convert starting_price to numeric float
units["starting_price"] = pd.to_numeric(units["starting_price"], errors="coerce")

units.head()

# 7. Analysis & answering questions

## Q1. Which property features drive rent?
Do bedrooms, bathrooms, square footage and property type change the monthly rent, and by how much?

In [ ]:
#Common homes only: studio to 4 bedrooms, 1 to 4 bathrooms
common = listings[(listings["beds"] <= 4) & (listings["baths"] <= 4)]

rent_grid = common.pivot_table(index="beds", columns="baths", values="rent", aggfunc="median")
count_grid = common.pivot_table(index="beds", columns="baths", values="rent", aggfunc="count")

rent_grid = rent_grid.where(count_grid >= 10)   # hide combinations with fewer than 10 listings
rent_grid.index = rent_grid.index.astype(int)
rent_grid.columns = rent_grid.columns.astype(int)
rent_grid

In [ ]:
plt.figure(figsize=(7, 4))
sns.heatmap(rent_grid, annot=True, fmt=",.0f", cmap="Blues")
plt.title("Median monthly rent ($) by bedrooms and bathrooms")
plt.xlabel("Bathrooms")
plt.ylabel("Bedrooms (0 = studio)")
plt.show()

Size and property type in numbers:

In [ ]:
listings["size_group"] = pd.cut(listings["area"], [0, 700, 1000, 1500, 2500, 20000],
                                labels=["under 700", "700-1,000", "1,000-1,500", "1,500-2,500", "over 2,500"])

listings.groupby("size_group")[["rent", "rent_per_sqft"]].median()

In [ ]:
mid_size = listings[listings["area"].between(800, 1200)]   # homes of similar size

pd.DataFrame({
    "all listings": listings.groupby("type")["rent"].median(),
    "800-1,200 sq ft only": mid_size.groupby("type")["rent"].median(),
})

**Answer 1:** Rent rises with every bedroom, bathroom, and square foot. 

- With 2 bathrooms, going from 2 to 3 to 4 bedrooms moves the median from `$2,900` to `$4,000` to about `$4,970`.
- One extra bathroom adds roughly `$350` to `$1,000` at the same bedroom count (up to 3 bathrooms).
- Bigger homes cost more in total, but less per sq ft (`$3.78` under 700 sq ft vs `$2.49` over 2,500 sq ft).
- Property type matters on top of size: among homes of 800 to 1,200 sq ft, a house rents for about `$3,750` vs `$2,650` for an apartment.

## Q2. Which cities are the most expensive once home size is accounted for?
Where does a high rent just mean bigger homes? We compare **median rent** with **median rent per sq ft**.

In [ ]:
city_table = listings.groupby("city").agg(
    listings=("rent", "count"),
    median_rent=("rent", "median"),
    median_per_sqft=("rent_per_sqft", "median"),
)

big_cities = city_table[city_table["listings"] >= 25].copy()   # a city with 3 listings gives an unreliable median

big_cities["rank_by_rent"] = big_cities["median_rent"].rank(ascending=False)
big_cities["rank_by_sqft"] = big_cities["median_per_sqft"].rank(ascending=False)

big_cities.sort_values(by="rank_by_rent").head(5)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(big_cities["median_rent"], big_cities["median_per_sqft"], s=60, alpha=0.7)

# label only some cities, otherwise the names pile up on top of each other
for city in ["Corona Del Mar", "Newport Beach", "Laguna Beach", "Irvine", "Costa Mesa",
             "Yorba Linda", "Fountain Valley", "Brea", "Buena Park", "Huntington Beach"]:
    row = big_cities.loc[city]
    plt.annotate(city, (row["median_rent"], row["median_per_sqft"]), xytext=(6, 5), textcoords="offset points")

plt.title("Rent vs rent per sq ft by city (cities with 25+ listings)")
plt.xlabel("Median monthly rent ($)")
plt.ylabel("Median rent per sq ft ($)")
plt.show()

## Q3. How much more does a coastal address cost for the same home?
We compare coastal and inland listings **with the same number of bedrooms**, so we are not comparing big homes with small ones.

In [ ]:
coastal_cities = ["Huntington Beach", "Seal Beach", "Newport Beach", "Corona Del Mar", "Newport Coast",
                  "Laguna Beach", "Dana Point", "San Clemente", "Capistrano Beach", "Sunset Beach", "Surfside"]

listings["coastal"] = listings["city"].isin(coastal_cities)
listings["coastal"].value_counts()

In [ ]:
#Compare coastal vs inland for the same number of bedrooms
coast_table = listings[listings["beds"].between(1, 4)].pivot_table(index="beds", columns="coastal", values="rent", aggfunc="median")
coast_table.columns = ["Inland", "Coastal"]
coast_table["difference ($)"] = coast_table["Coastal"] - coast_table["Inland"]
coast_table["premium (%)"] = (coast_table["Coastal"] / coast_table["Inland"] - 1) * 100
coast_table.round(0)

In [ ]:
coast_table[["Inland", "Coastal"]].plot(kind="bar", color=["#2a78d6", "#eb6834"], title="Median rent: inland vs coastal")
plt.xlabel("Bedrooms")
plt.ylabel("Median monthly rent ($)")
plt.show()

In [ ]:
#Add per sq ft, which also removes the size difference
listings.groupby("coastal")["rent_per_sqft"].median()

**Answer 3:** 
- For the same bedrooms, coastal rent is always higher: about **`$640` more for a 1 bed (+31%), `$845` for a 2 bed (+31%), `$2,700` for a 3 bed (+66%) and `$2,200` for a 4 bed (+40%)**.
- Per sq ft the gap is `$3.71` on the coast vs `$2.88` inland (about +29%).

## Q4. How competitive is the market?
How long do listings sit, and when do landlords start cutting rent?

In [ ]:
#price_change is negative when the rent was reduced
listings["price_cut"] = listings["price_change"] < 0

listings["days_group"] = pd.cut(listings["days_on_zillow"], [-1, 7, 30, 60, 90, 10000],
                                labels=["0-7 days", "8-30 days", "31-60 days", "61-90 days", "90+ days"])

print("typical days on Zillow:", listings["days_on_zillow"].median())
print("share of all listings with a price cut:", round(listings["price_cut"].mean() * 100), "%")
print("typical cut ($):", listings[listings["price_cut"]]["price_change"].median())

cut_by_days = listings.groupby("days_group")["price_cut"].mean() * 100
cut_by_days.round(0)

In [ ]:
cut_by_days.plot(kind="bar", title="Share of listings with a price cut, by time on Zillow")
plt.xlabel("Days on Zillow")
plt.ylabel("% of listings with a price cut")
plt.show()

**Answer 4:** 
- The typical listing has been up 20 days, and 24% of all listings have had a price cut.
- Cuts are rare in the first week (4%), reach 20% by day 30, and jump to 41% for listings 31 to 60 days old, so landlords start cutting around the one-month mark. The typical cut is small, about `$150`.

## Q5. Are apartment buildings priced differently from individual rentals?
Buildings only list a *starting* price per unit type, so we compare them with individual apartments, condos and townhomes with the same number of bedrooms.

In [ ]:
individual = listings[listings["type"].isin(["APARTMENT", "CONDO", "TOWNHOUSE"])].groupby("beds")["rent"].median()
building = units.groupby("beds")["starting_price"].median()

compare = pd.DataFrame({"individual listings": individual, "building starting price": building}).loc[0:3]
compare["difference (%)"] = (compare["building starting price"] / compare["individual listings"] - 1) * 100
compare.round(0)

In [ ]:
compare[["individual listings", "building starting price"]].plot(kind="bar", color=["#2a78d6", "#eb6834"],
                                                                  title="Individual listings vs apartment building starting prices")
plt.xlabel("Bedrooms (0 = studio)")
plt.ylabel("Median monthly rent ($)")
plt.show()

**Answer 5:** Yes, buildings advertise higher prices for small units: **+13% for studios (`$2,340` vs `$2,070`), +14% for 1 beds (`$2,395` vs `$2,100`)**, then +5% for 2 beds and about equal for 3 beds. 

Part of the gap is that "starting from" is a minimum price that real leases can exceed, so treat building prices as an entry point, not a final rent.

# 8. Summary

**Cleaning**
- 300 columns reduced to 23, one reason at a time. 163 were photo links.
- The missing rent was not an error: 604 rows are apartment buildings with starting prices. We split the data into `listings` and `buildings`.
- Removed 4 mobile home rows with sale prices, fixed 1 impossible area, fixed "Dana Pt" and "Dana Point", converted dates and `$` text to real types.
- Kept luxury rentals (real data) and used the **median** instead of the mean.

**Answers**
1. **Features:** Rent rises with bedrooms, bathrooms and size. Bigger homes cost less per sq ft, and a house costs more than an apartment of the same size.
2. **Cities:** Corona Del Mar, Newport Beach and Laguna Beach are expensive even per sq ft. Irvine looks expensive but is average per sq ft.
3. **Coast:** For the same bedrooms, coastal rent is `31%` to `66%` higher (`$640` to `$2,700` a month).
4. **Market speed:** Typical listing is up 20 days. Price cuts jump from `20%` to `41%` of listings after day 30.
5. **Buildings:** Buildings advertise `13%` to `14%` more than individual rentals for studios and 1 beds, and the gap closes by 3 beds.

**Limits**
- One snapshot from 2025 of asking rents, not signed leases.
- Some cities have few listings, so we only compared cities with 25 or more.
- Building prices are "starting from" prices.
- This data has no amenities such as parking or laundry.

# 9. Save the clean data
Save the cleaned listings as a CSV so we can open it in Excel, Power BI or Tableau.

In [ ]:
export = listings.drop(columns=["is_building", "building_name"] + [c for c in listings.columns if c.startswith("units/")])

# a text label for bedrooms makes nicer charts in Power BI / Tableau
export["beds_group"] = export["beds"].clip(upper=5).map({0: "Studio", 1: "1 bed", 2: "2 beds", 3: "3 beds", 4: "4 beds", 5: "5+ beds"})

export.to_csv("listings_clean.csv", index=False)
units.to_csv("building_units_clean.csv", index=False)
export.shape